---
short_title: "Dateien öffnen"
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Dateien öffnen

Alle Programme, die wir bisher geschrieben haben, verlieren ihre Daten, sobald
sie beendet werden. Variablen im Notebook oder in der REPL existieren nur so
lange, wie die Python-Sitzung läuft. Wollen wir Daten **dauerhaft** speichern
oder Daten austauschen, die jemand anderes erstellt hat, kommen **Dateien** ins
Spiel.

In diesem Kapitel lernen wir, wie wir Text- und {term}`CSV`-Dateien in Python öffnen und
deren Inhalt lesen.

## Warum Dateien?

Dateien sind der einfachste Weg, Daten zu speichern. Sie sind

- **dauerhaft**: Der Inhalt bleibt nach dem Beenden des Programms erhalten.
- **austauschbar**: Andere Programme und Personen können dieselbe Datei lesen.
- **nachvollziehbar**: Bei Textdateien ist der Inhalt direkt lesbar.

In Bibliotheken begegnen uns Dateien ständig: CSV-Exporte aus
Bibliothekssystemen, Katalogdaten, Protokolldateien oder Konfigurationsdateien.

Bevor wir uns dem speziellen Format CSV zuwenden, schauen wir uns den
allgemeinen Umgang mit Dateien an.

## Eine Datei mit `open()` öffnen

Die Funktion `open()` öffnet eine Datei und gibt ein **Dateiobjekt** zurück.
Gelesen wird anschließend über dieses Objekt.

```python
datei = open("daten.csv", encoding="utf-8")
inhalt = datei.read()
datei.close()
```

Wichtig sind drei Angaben:

1. **Pfad**: Wo liegt die Datei? (relativ oder absolut)
2. **Modus**: Wollen wir lesen (`"r"`), schreiben (`"w"`) oder anhängen (`"a"`)?
   Ohne Angabe wird gelesen.
3. **Encoding**: Wie sind die Zeichen codiert? Für uns fast immer `"utf-8"`.

In [1]:
# Den Pfad zur Beispieldatei merken wir uns in einer Variablen.
# Der Pfad ist *relativ*: Er wird vom Ordner dieses Notebooks aus gelesen.
datei = "../assets/data/books_powerlaw_dataset.csv"

f = open(datei, encoding="utf-8")
print(type(f))

# `read(60)` liest die ersten 60 Zeichen.
print(f.read(60))

# Nach dem Lesen schließen wir die Datei wieder.
f.close()

<class '_io.TextIOWrapper'>
isbn,author,year,title,sales_year,sales
9780241951446,"Murak


## Relative und absolute Pfade

Ein **relativer Pfad** beschreibt den Ort einer Datei ausgehend vom aktuellen
Arbeitsverzeichnis. Ein **absoluter Pfad** beginnt beim Wurzelverzeichnis des
Dateisystems (unter {term}`Unix` bei `/`).

Unser Datensatz liegt im Ordner `assets/data/`, der eine Ebene über diesem
Kapitel liegt. Daher lautet der relative Pfad
`../assets/data/books_powerlaw_dataset.csv`. Die beiden Punkte `..` stehen für
den übergeordneten Ordner.

In [2]:
import os

print("Aktuelles Arbeitsverzeichnis:", os.getcwd())
print("Existiert die Datei?", os.path.exists(datei))
print("Absoluter Pfad:", os.path.abspath(datei))

Aktuelles Arbeitsverzeichnis: /Users/h_schnaitter/code/Selbstlernkurs_Python/040-Projekt_CSV_I
Existiert die Datei? True
Absoluter Pfad: /Users/h_schnaitter/code/Selbstlernkurs_Python/assets/data/books_powerlaw_dataset.csv


## Methoden zum Lesen

Ein Dateiobjekt kennt verschiedene Methoden, um Inhalte zu lesen:

| Methode | Ergebnis |
| --- | --- |
| `read()` | die gesamte Datei als ein String |
| `read(n)` | die nächsten `n` Zeichen |
| `readline()` | die nächste Zeile (inklusive Zeilenumbruch) |
| `readlines()` | alle Zeilen als Liste von Strings |

Beim Lesen merkt sich das Dateiobjekt, wie weit es bereits gelesen hat. Man
spricht vom **Dateizeiger**. Nach dem Öffnen steht er am Anfang der Datei.

In [3]:
f = open(datei, encoding="utf-8")

# `repr()` macht den Zeilenumbruch `\n` am Ende sichtbar.
print(repr(f.readline()))   # Header-Zeile
print(repr(f.readline()))   # erste Datenzeile

f.close()

'isbn,author,year,title,sales_year,sales\n'
'9780241951446,"Murakami, Haruki",1984,Norwegian Wood,2020,61\n'


In [4]:
with open(datei, encoding="utf-8") as f:
    zeilen = f.readlines()

print("Anzahl Zeilen (inklusive Header):", len(zeilen))
print("Erste Zeile:", zeilen[0].strip())
print("Letzte Zeile:", zeilen[-1].strip())

Anzahl Zeilen (inklusive Header): 181
Erste Zeile: isbn,author,year,title,sales_year,sales
Letzte Zeile: 9783518293454,"Luhmann, Niklas",1984,Soziale Systeme,2025,5


## Der `with`-Kontextmanager

Eine Datei muss geschlossen werden, damit die Ressource freigegeben wird und
alle Schreibvorgänge abgeschlossen sind. Vergisst man `close()`, kann es zu
Problemen kommen. Deshalb verwendet man in Python fast immer das Schlüsselwort
`with`.

Der `with`-Block sorgt dafür, dass die Datei **automatisch geschlossen** wird –
auch dann, wenn im Block ein Fehler auftritt.

In [5]:
with open(datei, encoding="utf-8") as f:
    inhalt = f.read()

print("Zeichen gelesen:", len(inhalt))

# Nach dem with-Block ist die Datei automatisch geschlossen.
print("Datei geschlossen?", f.closed)

Zeichen gelesen: 11852
Datei geschlossen? True


## Encoding (Zeichenkodierung)

Ein Computer speichert Text als Folge von Bytes. Das **Encoding** legt fest,
wie diese Bytes auf Zeichen abgebildet werden. Umlaute wie `ä`, `ö`, `ü` und
das scharfe `ß` belegen je nach Encoding unterschiedlich viele Bytes.

Für moderne Textdateien gilt: `encoding="utf-8"`. UTF-8 kann alle Zeichen
darstellen, auch nicht-lateinische Schriften.

```{note}
Öffnen wir eine UTF-8-Datei mit dem falschen Encoding, erhalten wir einen
`UnicodeDecodeError`. Geben wir umgekehrt kein Encoding an, nutzt Python die
systemabhängige Standardkodierung. Deshalb geben wir `encoding="utf-8"` immer
explizit an.
```

In [6]:
try:
    with open(datei, encoding="ascii") as f:
        f.read()
except UnicodeDecodeError as fehler:
    print("UnicodeDecodeError aufgetreten:")
    print(fehler)

UnicodeDecodeError aufgetreten:
'ascii' codec can't decode byte 0xc3 in position 715: ordinal not in range(128)


## Typische Fehler

### Datei nicht gefunden

Existiert die angegebene Datei nicht, löst `open()` einen `FileNotFoundError`
aus. Häufige Ursachen sind ein falscher Pfad, Tippfehler oder ein Programm, das
aus einem anderen Ordner gestartet wird.

In [7]:
try:
    with open("diese_datei_gibt_es_nicht.csv", encoding="utf-8") as f:
        f.read()
except FileNotFoundError as fehler:
    print("FileNotFoundError aufgetreten:")
    print(fehler)

FileNotFoundError aufgetreten:
[Errno 2] No such file or directory: 'diese_datei_gibt_es_nicht.csv'


### Falsches Encoding

Beim `UnicodeDecodeError` nennt Python das problematische Byte und die Position.
Das hilft, das richtige Encoding zu finden.

## Zusammenfassung

- `open(pfad, encoding="utf-8")` öffnet eine Datei zum Lesen.
- Der `with`-Block schließt die Datei automatisch.
- `read()`, `readline()` und `readlines()` lesen unterschiedlich viel.
- Denken Sie immer an das richtige Encoding (`utf-8`).
- `FileNotFoundError` und `UnicodeDecodeError` sind typische Fehlerquellen.

```{hint}
Im nächsten Kapitel zerlegen wir den gelesenen Text „von Hand“ in die einzelnen
Spalten einer CSV-Datei.
```